In [3]:
from pprint import PrettyPrinter
import pandas as pd

# Create a PrettyPrinter instance with "indent=2"
pp = PrettyPrinter(indent=2)

In [6]:
from pymongo import MongoClient
from load_mongo_data import load_nairobi_to_mongodb
host="192.119.237.3"
port=27017

client = MongoClient(host=host, port=port)
# Connect to MongoDB on localhost
load_nairobi_to_mongodb(host=host)

✅ Nairobi data successfully loaded to MongoDB at 192.119.237.3:27017!
   Collection: air-quality.nairobi | Documents inserted: 10000


In [7]:
# Get list of databases
client = MongoClient(f"mongodb://{host}:{port}")

database_list = client.list_database_names()

# Print using pretty printer
pp.pprint(database_list)

['admin', 'air-quality', 'config', 'local', 'wqu-abtest']


In [22]:
# Access the air_quality database
db = client["air_quality"]

In [23]:
# Get list of collections
collection_list = db.list_collection_names()

# Print the collections
print(collection_list)

[]


In [24]:
# Access the nairobi collection
nairobi = db['nairobi']

In [25]:
# Count total documents
total_documents = nairobi.count_documents({})

print(f"Total documents: {total_documents:,}")

Total documents: 0


In [26]:
# Get one sample document
sample_document = nairobi.find_one()

# Print it
pp.pprint(sample_document)

None


In [27]:
# Get distinct sensor IDs
unique_sensors = nairobi.distinct("sensor_id")

print(f"Number of unique sensors: {len(unique_sensors)}")
print(f"First 5 sensors: {unique_sensors[:5]}")

Number of unique sensors: 0
First 5 sensors: []


In [28]:
measurement_types = nairobi.distinct("value_type")
print(f"Measurement types: {measurement_types}")

Measurement types: []


In [29]:
# Create aggregation pipeline
pipeline = [
    {"$group": {"_id": "$sensor_id", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 5}
]

# Execute aggregation
sensor_reading_counts = list(nairobi.aggregate(pipeline))

# Print results
pp.pprint(sensor_reading_counts)

[]


In [30]:
# Get the first sensor ID
if unique_sensors:
    first_sensor = unique_sensors[0]
else:
    print("No sensors found!")

# Create pipeline to count by type for this sensor
pipeline = [
    {"$match": {"sensor_id": first_sensor}},
    {"$group": {"_id": "...", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]

sensor_type_counts = list(nairobi.aggregate(pipeline))
pp.pprint(sensor_type_counts)

No sensors found!
[]


In [31]:
# Query PM2.5 readings from first sensor
pm25_cursor = nairobi.find(
    {"sensor_id": first_sensor, "value_type": "P2"},
    projection={"timestamp": True, "value": True, "_id": False}
).limit(5)

# Convert to list and print
pm25_sample = list(pm25_cursor)
pp.pprint(pm25_sample)

[]


In [32]:
# Convert to DataFrame
df_raw = (
    pd.DataFrame(list(nairobi.find(
        {"sensor_id": first_sensor, "value_type": 'P2'},
        projection={'_id':0, 'timestamp':1, 'value':1}
    )))
    .assign(timestamp=lambda x: pd.to_datetime(x['timestamp']))
    .set_index('timestamp')
    .sort_index()
)

print(df_raw.head())
print(f"\nShape: {df_raw.shape}")

KeyError: 'timestamp'

In [33]:
# Check for missing values
print("Missing values:")
print(df_raw.isnull().sum())

print("\nBasic statistics:")
print(df_raw.describe())

Missing values:
value    0
dtype: int64

Basic statistics:
            value
count  953.000000
mean    39.755792
std      9.816934
min      8.030000
25%     33.100000
50%     39.890000
75%     46.360000
max     67.410000


In [34]:
db = "air-quality"
collection = "nairobi"
# host = <use the host value above>

def wrangle_data(
    db, 
    collection, 
    host=host, 
    port=27017
    ):
    """
    Retrieve and clean Nairobi air quality data from MongoDB.
    
    Parameters:
    -----------
    host : str
        MongoDB host address (default: localhost)
    port : int
        MongoDB port (default: 27017)
    
    Returns:
    --------
    pd.DataFrame
        Clean time series data indexed by timestamp with 'pm25' column
    """
    from pymongo import MongoClient
    
    # Connect to MongoDB
    client = MongoClient(f"mongodb://{host}:{port}")
    
    # Query and clean data using method chaining
    return (
        pd.DataFrame(
            list(
                client[db][collection]
                .find({"value_type": "P2"}, 
                      projection={"value": 1, "timestamp": 1, "_id": 0})
                .sort("timestamp", 1)
            )
        )
        # Convert timestamp to datetime
        .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
        # Set timezone (data is in UTC, convert to Nairobi time)
        .assign(timestamp=lambda x: x["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Africa/Nairobi"))
        # Set index
        .set_index("timestamp")
        # Handle missing values
        .dropna()
        # Sort chronologically
        .sort_index()
        # Rename column for clarity
        .rename(columns={"value": "pm25"})
    )
    


# Test the function
df_clean = wrangle_data(db, collection, host)
print(df_clean.head())
print(f"\nShape: {df_clean.shape}")

                            pm25
timestamp                       
2024-01-01 03:00:00+03:00  45.42
2024-01-01 04:00:00+03:00  28.71
2024-01-01 05:00:00+03:00  30.34
2024-01-01 06:00:00+03:00  28.35
2024-01-01 07:00:00+03:00  52.78

Shape: (10000, 1)


In [35]:
%%writefile mongo_wrangle.py
"""
MongoDB Data Wrangling Module for Nairobi Air Quality Data

This module provides functions to retrieve and clean air quality data
from MongoDB for time series analysis.
"""

import pandas as pd
from pymongo import MongoClient


def wrangle_data(
    db, 
    collection, 
    host, 
    port=27017
    ):
    
    """
    Retrieve and clean Nairobi air quality data from MongoDB.
    
    Parameters:
    -----------
    host : str
        MongoDB host address (default: localhost)
    port : int
        MongoDB port (default: 27017)
    
    Returns:
    --------
    pd.DataFrame
        Clean time series data indexed by timestamp with 'pm25' column
    """
    # Connect to MongoDB
    client = MongoClient(f"mongodb://{host}:{port}")
    
    # Query and clean data using method chaining
    return(
        pd.DataFrame(
            list(
                client[db][collection]
                .find({"value_type": "P2"}, 
                      projection={"value": 1, "timestamp": 1, "_id": 0})
                .sort("timestamp", 1)
            )
        )
        # Convert timestamp to datetime
        .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
        # Set timezone (data is in UTC, convert to Nairobi time)
        .assign(timestamp=lambda x: x["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Africa/Nairobi"))
        # Set index
        .set_index("timestamp")
        # Handle missing values
        .dropna()
        # Sort chronologically
        .sort_index()
        # Rename column for clarity
        .rename(columns={"value": "pm25"})
    )

Overwriting mongo_wrangle.py
